# Reference: the shape of the a priori hierarchy

The directory tree under `images/<variant>/` **is** the Kiani-Mur semantic hierarchy: `animate/animal/body/bird/duck3.png` encodes domain, then mid-level category, then basic level. `D_sem_km` is the path length through that tree, and `s0_build_rdms/validate_rdms.py` checks that it reproduces the tree exactly.

This notebook looks only at the tree, before anything is compared with it. It is one of two observational notebooks: `0b_joint_behavior` describes the joint behavioral RDM in the same way. Notebooks `1_` to `6_` then ask how similar each RDM is to the tree:

| # | question |
|---|---|
| `0b_joint_behavior` | (observational) where are the behavioral ratings missing, how noisy are they, how do they relate to external data and to semantic and sensory predictors? |
| `1_mds_dimensionality` | which MDS embeddings of the behavioral RDM to carry forward, and does embedding move it toward the tree? |
| `2_rank_agreement` | how much rank structure does each RDM share with the tree? |
| `3_depth_gradient` | does distance grow as two images diverge higher up the tree? |
| `4_level_by_level` | which levels of the tree does each RDM recover? |
| `5_semantic_vs_lowlevel` | is the agreement semantic, or inherited from pixel similarity? |
| `6_tree_recovery` | does each RDM's own clustering reproduce the tree? |

**Run from this directory** with the RDMs built (`python -m analysis.s0_build_rdms.build_all --validate` from the repo root).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/s2_semantic_hierarchy/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import plotly.io as pio

pio.renderers.default = "notebook_connected"  # figures load plotly.js once instead of embedding it

import numpy as np
import pandas as pd

from analysis.s2_semantic_hierarchy import common as c
from analysis.s2_semantic_hierarchy import tree_shape as ts
from analysis.utils.hierarchy import depth_is_ragged

PARTS = c.tree_parts()
KM = c.load_rdm_set([c.REFERENCE])[c.REFERENCE]
print(f"{len(PARTS)} images, {len(KM):,} pairs; ragged tree: {depth_is_ragged(PARTS)}")

725 images, 262,450 pairs; ragged tree: True


## 1. How is the tree built?

Categories per level, and how unevenly the images are spread over them.

In [2]:
ts.fig_tree(PARTS).show()
print(ts.categories_per_level(PARTS).to_string())

                               name  n_categories  min_images  median_images  max_images
level                                                                                   
1      domain (animate / inanimate)             2         298          362.5         427
2                mid-level category             4         134          162.0         267
3                       basic level            16           6           38.5         102


## 2. Is the tree ragged, and what does that do to KM distance?

If every leaf sat at one depth, KM distance would be a function of the LCA depth alone (how far up the two images diverge). It is not: leaves sit at depths 3, 4 and 5, and a pair in a finely subdivided branch gets a larger KM distance than an equally related pair in a coarse branch.

In [3]:
print(ts.leaf_depth_counts(PARTS).to_string())
print("\nKM distance (columns) per LCA depth (rows), pair counts:")
print(ts.km_vs_lca(KM, PARTS).to_string())

            n_images pooled_tier
leaf_depth                      
3                336           3
4                316         >=4
5                 73         >=4

KM distance (columns) per LCA depth (rows), pair counts:
km_distance     1      2      3      4      5      6      7      8     9
lca_depth                                                               
0               0      0      0      0      0      0  75600  45003  6643
1               0      0      0  21983  19317  11424  11972      0     0
2               0  27596  11059   6834   2493   1290      0      0     0
3            6901   7269   1960      0      0      0      0      0     0
4            3768    990      0      0      0      0      0      0     0
5             348      0      0      0      0      0      0      0     0


Every LCA depth maps to several KM distances: the raggedness made concrete. Level-wise comparisons in the later notebooks are therefore read within strata of branch depth, as the pre-registration specifies.

## 3. Pooling the deep tree

Depths 4 and 5 both lie below the basic level, and depth 5 is small, so all later notebooks pool them into one tier (`>=4`):

* leaf depth is binned as {3, >=4};
* LCA depth is binned as {0, 1, 2, 3, >=4};
* a pair's **branch-depth stratum** is the unordered pair of its two binned leaf depths: 3x3, 3x>=4, >=4x>=4.

Where the deep tier comes from:

In [4]:
print(ts.deepest_tier_images(PARTS).to_string())
domain_by_tier = pd.crosstab(pd.Series([p[0] for p in PARTS], name="domain"),
                             pd.Series([c.tier_label(d) for d in c.pooled_leaf_depths(PARTS)], name="leaf tier"))
print("\n" + domain_by_tier.to_string())

                            n_images
animate/human/face               102
animate/animal/body               81
inanimate/natural/flower          71
animate/human/body                62
animate/animal/face               53
inanimate/handmade/vehicle        20

leaf tier    3  >=4
domain             
animate      0  298
inanimate  336   91


In [5]:
counts = ts.pair_counts(PARTS)
ts.fig_pair_counts(counts).show()
print(counts.to_string())
print("\nKM distance per pooled LCA bin:")
print(ts.km_vs_lca(KM, PARTS, pooled=True).to_string())

stratum    3x3   3x>=4  >=4x>=4     all
lca_bin                                
0            0  100128    27118  127246
1        21983   19317    23396   64696
2        27596   11059    10617   49272
3         6701     200     9229   16130
>=4          0       0     5106    5106
all      56280  130704    75466  262450

KM distance per pooled LCA bin:
km_distance     1      2      3      4      5      6      7      8     9
lca_depth                                                               
0               0      0      0      0      0      0  75600  45003  6643
1               0      0      0  21983  19317  11424  11972      0     0
2               0  27596  11059   6834   2493   1290      0      0     0
3            6901   7269   1960      0      0      0      0      0     0
>=4          4116    990      0      0      0      0      0      0     0


**Branch depth and domain are almost the same split.** Every animate image sits at depth >= 4, and every depth-3 image is inanimate (91 of the 427 inanimate images, the flowers and vehicles, are deep too). So the 3x3 stratum holds only inanimate pairs, and its pairs never diverge at the root. Any stratified result therefore partly reflects the animate/inanimate split; later notebooks need to say so when they report strata.

**What pooling costs.** WordNet's distance profile reversed in the old notebook, entirely at LCA depth 5, driven by primate pairs (WordNet routes Old World to New World monkeys through the primate root, so `baboon x capuchin` scores far above `baboon x macaque`). Merged into the >= 4 bin (5,106 pairs, 348 of them at depth 5), that reversal is no longer visible. It is a WordNet bookkeeping quirk, not a property of the images, and it is recorded here rather than analysed further.

## 4. Takeaways

1. **The tree is small at the top and uneven below.** 2 domains, 4 mid-level and 16 basic-level categories; basic-level categories hold between 6 and 102 images (human faces).
2. **The tree is ragged**, so KM distance is not a function of LCA depth alone: each LCA depth maps to two to five KM values.
3. **Depths 4 and 5 are pooled** into a `>=4` tier everywhere, giving LCA bins {0, 1, 2, 3, >=4} and three branch-depth strata (3x3, 3x>=4, >=4x>=4).
4. **The strata are confounded with domain.** All animate images are deep and all depth-3 images are inanimate, so the 3x3 stratum is inanimate-only. Stratified results must be read with that in mind.
5. **Pooling hides WordNet's primate reversal** at LCA depth 5; it is noted here and not analysed further.